# Fashion-MNIST compression results

Figures for the blog post. Numbers come from `python scripts/run.py` (seed 42, CPU). This notebook only reads saved CSVs. It does not train.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, here.parent):
        if (candidate / "scripts" / "run.py").is_file():
            return candidate
    return here

ROOT = repo_root()
OUT = ROOT / "results" / "fashion_mnist"
comparison_path = OUT / "comparison.csv"
if not comparison_path.is_file():
    raise SystemExit(
        f"Missing {comparison_path}. From the repo root, run: python scripts/run.py"
    )
print(f"Reading {OUT}")
print("Numbers are from python scripts/run.py (seed 42, CPU).")

## Comparison table

Dynamic and static INT8 parameter counts are not comparable to the fp32 rows, so those cells are blank. Size and accuracy are the comparable columns.

In [ ]:
comparison = pd.read_csv(comparison_path)
table = comparison[
    ["name", "accuracy", "nonzero_params", "sparsity", "size_mb", "latency_p50_ms"]
].copy()
int8 = table["name"].isin(["teacher_dynamic_int8", "teacher_static_int8"])
table.loc[int8, ["nonzero_params", "sparsity"]] = pd.NA
table["accuracy"] = (table["accuracy"] * 100).round(2)
table["sparsity"] = (table["sparsity"] * 100).round(1)
table["size_mb"] = table["size_mb"].round(2)
table["latency_p50_ms"] = table["latency_p50_ms"].round(1)
table = table.rename(
    columns={
        "accuracy": "accuracy_%",
        "nonzero_params": "nonzero_params",
        "sparsity": "sparsity_%",
        "size_mb": "size_mb",
        "latency_p50_ms": "latency_p50_ms",
    }
)
table

## Teacher validation curve

The vertical line is the checkpoint the comparison used (`selected_epochs` in `run_summary.json`).

In [ ]:
import json

teacher = pd.read_csv(OUT / "teacher_curve.csv")
summary = json.loads((OUT / "run_summary.json").read_text(encoding="utf-8"))
selected = int(summary["selected_epochs"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(teacher["epoch"], teacher["val_accuracy"], marker="o")
ax.axvline(selected, color="0.4", linestyle="--", label=f"selected epoch {selected}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation accuracy")
ax.set_title("Teacher validation accuracy")
ax.legend()
fig.tight_layout()
plt.show()

## Student training loss

In [ ]:
student_ce = pd.read_csv(OUT / "student_ce_curve.csv")
student_kd = pd.read_csv(OUT / "student_kd_curve.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(student_ce["epoch"], student_ce["train_loss"], marker="o", label="student_ce")
ax.plot(student_kd["epoch"], student_kd["train_loss"], marker="o", label="student_kd")
ax.set_xlabel("Epoch")
ax.set_ylabel("Train loss")
ax.set_title("Student training loss")
ax.legend()
fig.tight_layout()
plt.show()

## Prune curve

The 0% point is the same one-epoch fine-tune with no weights zeroed.

In [ ]:
prune = pd.read_csv(OUT / "prune_sparsity_curve.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(prune["sparsity"] * 100, prune["accuracy"] * 100, marker="o")
control = prune.loc[prune["amount"] == 0].iloc[0]
ax.annotate(
    "extra-epoch control",
    xy=(control["sparsity"] * 100, control["accuracy"] * 100),
    xytext=(8, 8),
    textcoords="offset points",
)
ax.set_xlabel("Measured sparsity (%)")
ax.set_ylabel("Test accuracy (%)")
ax.set_title("Test accuracy vs sparsity")
fig.tight_layout()
plt.show()

## Per-class accuracy

In [ ]:
per_class = pd.read_csv(OUT / "per_class_accuracy.csv")
keep = ["teacher_fp32", "student_kd", "teacher_pruned", "teacher_static_int8"]
per_class = per_class[per_class["name"].isin(keep)].set_index("name").loc[keep]
plot_df = per_class.T * 100

fig, ax = plt.subplots(figsize=(9, 4))
plot_df.plot(kind="bar", ax=ax)
ax.set_ylabel("Test accuracy (%)")
ax.set_xlabel("")
ax.set_title("Per-class test accuracy")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## Class samples

In [ ]:
samples = OUT / "samples.png"
if samples.is_file():
    image = plt.imread(samples)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.imshow(image)
    ax.axis("off")
    ax.set_title("One training example per class")
    fig.tight_layout()
    plt.show()
else:
    print(f"No sample grid at {samples}")